In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/cleaned/taxi_cleaned.csv')

# Re-convert datetime (CSV doesn't preserve datetime type)
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

print(f"✅ Cleaned data loaded")
print(f"Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")

✅ Cleaned data loaded
Shape: 604,075 rows, 9 columns
Columns: ['id', 'vendor_id', 'pickup_datetime', 'passenger_count', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'store_and_fwd_flag']


In [2]:
# Extract hour from pickup_datetime (0-23)
df['hour_of_day'] = df['pickup_datetime'].dt.hour

print("✅ hour_of_day created")
print(f"Sample values: {df['hour_of_day'].head(5).tolist()}")
print(f"Range: {df['hour_of_day'].min()} to {df['hour_of_day'].max()}")
print(f"\nTop 5 busiest hours:")
print(df['hour_of_day'].value_counts().head())

✅ hour_of_day created
Sample values: [23, 23, 23, 23, 23]
Range: 0 to 23

Top 5 busiest hours:
hour_of_day
18    37674
19    37249
20    35153
21    34610
22    33480
Name: count, dtype: int64


In [3]:
# Extract day name from pickup_datetime
df['day_of_week'] = df['pickup_datetime'].dt.day_name()

print("✅ day_of_week created")
print(f"\nTrips per day:")
# Sort by day order not alphabetical
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
print(df['day_of_week'].value_counts().reindex(day_order))

✅ day_of_week created

Trips per day:
day_of_week
Monday       77748
Tuesday      84275
Wednesday    86832
Thursday     90427
Friday       93111
Saturday     90934
Sunday       80748
Name: count, dtype: int64


In [4]:
# 1 = Saturday or Sunday, 0 = weekday
# dt.dayofweek: Monday=0, Tuesday=1, ..., Saturday=5, Sunday=6
df['is_weekend'] = (df['pickup_datetime'].dt.dayofweek >= 5).astype(int)

print("✅ is_weekend created")
print(f"\nValue counts:")
print(df['is_weekend'].value_counts())
print(f"\n0 = Weekday, 1 = Weekend")
weekday_count = df[df['is_weekend'] == 0].shape[0]
weekend_count = df[df['is_weekend'] == 1].shape[0]
print(f"Weekday trips : {weekday_count:,} ({weekday_count/len(df)*100:.1f}%)")
print(f"Weekend trips : {weekend_count:,} ({weekend_count/len(df)*100:.1f}%)")

✅ is_weekend created

Value counts:
is_weekend
0    432393
1    171682
Name: count, dtype: int64

0 = Weekday, 1 = Weekend
Weekday trips : 432,393 (71.6%)
Weekend trips : 171,682 (28.4%)


In [5]:
# Haversine formula calculates straight-line distance between two GPS points
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Earth's radius in kilometers
    
    # Convert degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    return R * c

# Apply to every row
df['trip_distance_km'] = haversine_distance(
    df['pickup_latitude'],
    df['pickup_longitude'],
    df['dropoff_latitude'],
    df['dropoff_longitude']
)

# Round to 3 decimal places
df['trip_distance_km'] = df['trip_distance_km'].round(3)

print("✅ trip_distance_km created")
print(f"\nDistance statistics:")
print(f"Minimum  : {df['trip_distance_km'].min():.3f} km")
print(f"Maximum  : {df['trip_distance_km'].max():.3f} km")
print(f"Average  : {df['trip_distance_km'].mean():.3f} km")
print(f"Median   : {df['trip_distance_km'].median():.3f} km")

✅ trip_distance_km created

Distance statistics:
Minimum  : 0.000 km
Maximum  : 41.548 km
Average  : 3.413 km
Median   : 2.093 km


In [6]:
# A trip with 0 km distance is invalid — passenger got in and out immediately
rows_before = len(df)
df = df[df['trip_distance_km'] > 0]
rows_after = len(df)

print(f"✅ Zero distance trips removed")
print(f"Rows removed: {rows_before - rows_after:,}")
print(f"Rows remaining: {rows_after:,}")

✅ Zero distance trips removed
Rows removed: 2,413
Rows remaining: 601,662


In [7]:
# Categorize trips into Short, Medium, Long based on distance
# Based on NYC taxi typical trip patterns:
# Short  : 0 to 2 km    (nearby neighborhood trips)
# Medium : 2 to 10 km   (cross-borough trips)
# Long   : over 10 km   (airport runs, long distance)

def categorize_distance(km):
    if km <= 2:
        return 'Short'
    elif km <= 10:
        return 'Medium'
    else:
        return 'Long'

df['distance_bucket'] = df['trip_distance_km'].apply(categorize_distance)

print("✅ distance_bucket created")
print(f"\nTrip distribution:")
print(df['distance_bucket'].value_counts())
print(f"\nAs percentage:")
print((df['distance_bucket'].value_counts() / len(df) * 100).round(1))

✅ distance_bucket created

Trip distribution:
distance_bucket
Short     286389
Medium    279029
Long       36244
Name: count, dtype: int64

As percentage:
distance_bucket
Short     47.6
Medium    46.4
Long       6.0
Name: count, dtype: float64


In [8]:
print("=== FINAL DATASET WITH ENGINEERED FEATURES ===")
print(f"Total rows    : {len(df):,}")
print(f"Total columns : {df.shape[1]}")

print("\n=== ALL COLUMNS ===")
for col in df.columns:
    print(f"  {col:<25} {df[col].dtype}")

print("\n=== SAMPLE ROW ===")
print(df[['pickup_datetime','hour_of_day','day_of_week',
          'is_weekend','trip_distance_km','distance_bucket']].head(3))

=== FINAL DATASET WITH ENGINEERED FEATURES ===
Total rows    : 601,662
Total columns : 14

=== ALL COLUMNS ===
  id                        object
  vendor_id                 int64
  pickup_datetime           datetime64[ns]
  passenger_count           int64
  pickup_longitude          float64
  pickup_latitude           float64
  dropoff_longitude         float64
  dropoff_latitude          float64
  store_and_fwd_flag        int64
  hour_of_day               int32
  day_of_week               object
  is_weekend                int64
  trip_distance_km          float64
  distance_bucket           object

=== SAMPLE ROW ===
      pickup_datetime  hour_of_day day_of_week  is_weekend  trip_distance_km  \
0 2016-06-30 23:59:00           23    Thursday           0             2.746   
1 2016-06-30 23:59:00           23    Thursday           0             2.759   
2 2016-06-30 23:59:00           23    Thursday           0             1.306   

  distance_bucket  
0          Medium  
1         

In [9]:
# Overwrite the cleaned file with new engineered columns
output_path = '../data/cleaned/taxi_cleaned.csv'
df.to_csv(output_path, index=False)

print(f"✅ Updated file saved to: {output_path}")
print(f"Final shape: {df.shape[0]:,} rows, {df.shape[1]} columns")

✅ Updated file saved to: ../data/cleaned/taxi_cleaned.csv
Final shape: 601,662 rows, 14 columns
